# Overnight WTI Baseline Runner

This notebook runs the full overnight WTI baseline batch and saves outputs to Google Drive or local Colab storage.

Batch included by default:
- univariate daily
- univariate weekly
- multivariate daily
- multivariate weekly

Models included by default:
- GRU
- TimeXer
- iTransformer

Each run saves:
- train / validation loss history CSV
- loss curve PNG
- forecast plot PNG
- metrics CSV
- predictions CSV
- per-run config snapshot
- batch summary CSV / HTML / Markdown report


In [1]:
%pip -q install "git+https://github.com/Nixtla/neuralforecast.git" pandas matplotlib openpyxl pyyaml


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 48.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you

In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jaeho777/newoil.git"
WORKDIR = Path("/content/newoil")
BATCH_CONFIG_RELATIVE_PATH = "configs/batches/overnight_wti_baseline.yaml"

SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/newoil_outputs")
LOCAL_OUTPUT_ROOT = Path("/content/newoil_outputs")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)
sys.path.insert(0, str(WORKDIR / "src"))

from newoil import run_batch_from_config

repo_root = WORKDIR
batch_config_path = repo_root / BATCH_CONFIG_RELATIVE_PATH
output_root = DRIVE_OUTPUT_ROOT if SAVE_TO_GOOGLE_DRIVE else LOCAL_OUTPUT_ROOT
output_root.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {repo_root}")
print(f"Batch config: {batch_config_path}")
print(f"Output root: {output_root}")


Mounted at /content/drive
Repo root: /content/newoil
Batch config: /content/newoil/configs/batches/overnight_wti_baseline.yaml
Output root: /content/drive/MyDrive/newoil_outputs


In [ ]:
result = run_batch_from_config(
    batch_config_path=batch_config_path,
    repo_root=repo_root,
    output_root=output_root,
)

summary_df = result.summary_df.copy()
summary_df


[RUN] uni_daily__GRU


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ hist_encoder │ GRU           │  363 K │ train │     0 │
│ 4 │ mlp_decoder  │ MLP           │ 25.9 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 388 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 388 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [ ]:
from IPython.display import Image, Markdown, display
import pandas as pd

display(Markdown(f"# Overnight Report\n\n- Batch dir: `{result.batch_dir}`\n- Summary CSV: `{result.batch_dir / 'summary.csv'}`\n- HTML report: `{result.report_html}`"))
display(summary_df)

for _, row in summary_df.iterrows():
    display(Markdown(f"## {row['run_name']}"))
    if row['status'] == 'failed':
        display(Markdown(f"Failed: `{row.get('error_message', '')}`"))
        continue

    artifact_dir = Path(row['artifact_dir'])
    metrics_df = pd.read_csv(artifact_dir / 'metrics.csv')
    display(metrics_df)
    display(Image(filename=str(artifact_dir / 'loss_curve.png')))
    display(Image(filename=str(artifact_dir / 'forecast_plot.png')))
